# Phase 1 Exhaustive EDA (Base Dataset)

This notebook runs an exhaustive diagnostic EDA pass for the validated Phase 1 dataset.

- Project: F1 pit stop strategy (2022-2024 Ground Effect Era)
- Dataset target: `base_df.csv` (expected 74,536 rows x 55 columns)
- Output artifacts:
  - Figures saved to `data/eda_outputs_phase1_base_df/figures/`
  - Plain-text report saved to `data/eda_outputs_phase1_base_df/phase1_eda_report.md`

Mandatory cast policy is applied immediately after load, before any filtering/grouping:
- `is_strategic_stop` -> `pd.BooleanDtype()`
- `sc_active`, `vsc_active`, `caution_active` -> `bool`

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 200)

# ---------------------------------------------------------------------
# Path discovery (exact existing location, no assumptions)
# ---------------------------------------------------------------------
root = Path.cwd().resolve()
if root.name != 'Code Base':
    if (root / 'data').exists() and (root / 'src').exists():
        pass
    elif root.parent.name == 'Code Base':
        root = root.parent

data_dir = root / 'data'
processed_dir = data_dir / 'processed'
base_matches = sorted(data_dir.rglob('base_df.csv'))
if not base_matches:
    raise FileNotFoundError('No base_df.csv found under data/ directory')

preferred = processed_dir / 'base_df.csv'
base_path = preferred if preferred in base_matches else base_matches[0]

output_dir = data_dir / 'eda_outputs_phase1_base_df'
fig_dir = output_dir / 'figures'
output_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

report_path = output_dir / 'phase1_eda_report.md'

print(f'Located base_df.csv: {base_path}')
print(f'Figure output folder: {fig_dir}')
print(f'Report output path: {report_path}')

# ---------------------------------------------------------------------
# Mandatory casts BEFORE any filtering/grouping
# ---------------------------------------------------------------------
df = pd.read_csv(base_path)

for c in ['is_strategic_stop', 'sc_active', 'vsc_active']:
    if c not in df.columns:
        raise KeyError(f'Mandatory cast column missing from base_df: {c}')

# Phase 1 base_df anomaly handling: caution_active may not be serialized in closed dataset.
if 'caution_active' not in df.columns:
    print('ANOMALY: caution_active missing in base_df.csv; deriving as (sc_active | vsc_active) for EDA only.')
    with pd.option_context('future.no_silent_downcasting', True):
        sc_bool = df['sc_active'].infer_objects(copy=False).fillna(False).astype(bool)
        vsc_bool = df['vsc_active'].infer_objects(copy=False).fillna(False).astype(bool)
    df['caution_active'] = sc_bool | vsc_bool

df['is_strategic_stop'] = df['is_strategic_stop'].astype(pd.BooleanDtype())
for c in ['sc_active', 'vsc_active', 'caution_active']:
    with pd.option_context('future.no_silent_downcasting', True):
        df[c] = df[c].infer_objects(copy=False).fillna(False).astype(bool)

print('Cast dtypes:')
print(df[['is_strategic_stop', 'sc_active', 'vsc_active', 'caution_active']].dtypes)
print(f'Loaded shape: {df.shape[0]} x {df.shape[1]}')
if df.shape != (74536, 55):
    print(f'WARNING: expected 74536 x 55, found {df.shape[0]} x {df.shape[1]}')

Located base_df.csv: /Users/demi/Downloads/Dissertation Code Project/Code Base/data/processed/base_df.csv
Figure output folder: /Users/demi/Downloads/Dissertation Code Project/Code Base/data/eda_outputs_phase1_base_df/figures
Report output path: /Users/demi/Downloads/Dissertation Code Project/Code Base/data/eda_outputs_phase1_base_df/phase1_eda_report.md
ANOMALY: caution_active missing in base_df.csv; deriving as (sc_active | vsc_active) for EDA only.
Cast dtypes:
is_strategic_stop    boolean
sc_active               bool
vsc_active              bool
caution_active          bool
dtype: object
Loaded shape: 74536 x 56


In [3]:
# ---------------------------------------------------------------------
# Helpers for markdown report writing
# ---------------------------------------------------------------------
report_lines = []

def add_h1(text):
    report_lines.append(f'# {text}')
    report_lines.append('')

def add_h2(text):
    report_lines.append(f'## {text}')
    report_lines.append('')

def add_h3(text):
    report_lines.append(f'### {text}')
    report_lines.append('')

def add_p(text):
    report_lines.append(text)
    report_lines.append('')

def add_table(df_table, float_fmt='%.6f'):
    report_lines.append('```')
    if isinstance(df_table, pd.Series):
        report_lines.append(df_table.to_string())
    else:
        report_lines.append(df_table.to_string(float_format=lambda x: float_fmt % x if pd.notna(x) else 'nan'))
    report_lines.append('```')
    report_lines.append('')

add_h1('Phase 1 Exhaustive EDA Report')
add_p(f'Exact base_df.csv path used: {base_path}')
add_p(f'Loaded shape: {df.shape[0]} x {df.shape[1]}')
add_p('Mandatory casts applied before any filtering/grouping: is_strategic_stop -> BooleanDtype, sc_active/vsc_active/caution_active -> bool.')

# ---------------------------------------------------------------------
# 1) Descriptive statistics for six engineered feature groups
# ---------------------------------------------------------------------
feature_cols = [
    'deg_rate_corrected',
    'pit_window_delta',
    'undercut_threat_index',
    'overcut_threat_index',
    'circuit_sc_rate',
    'circuit_vsc_rate',
    'pace_differential',
    'position_at_stake',
]
missing_feature_cols = [c for c in feature_cols if c not in df.columns]
if missing_feature_cols:
    raise KeyError(f'Missing expected engineered feature columns: {missing_feature_cols}')

desc_rows = []
for col in feature_cols:
    s = pd.to_numeric(df[col], errors='coerce')
    desc_rows.append({
        'feature': col,
        'mean': s.mean(skipna=True),
        'std': s.std(skipna=True),
        'min': s.min(skipna=True),
        'max': s.max(skipna=True),
        'missing_pct': s.isna().mean() * 100.0,
    })

desc_df = pd.DataFrame(desc_rows).sort_values('feature').reset_index(drop=True)
print('Descriptive statistics for engineered feature set:')
print(desc_df)

add_h2('1. Descriptive statistics for engineered feature set')
add_p('Note: undercut/overcut threat index and SC/VSC circuit exposure rate each map to two concrete columns and are reported separately.')
add_table(desc_df)

# ---------------------------------------------------------------------
# 2) Pit stop lap distribution split by is_strategic_stop
# ---------------------------------------------------------------------
pit_rows = df[df['is_strategic_stop'].notna()].copy()
if pit_rows.empty:
    raise ValueError('No pit rows found where is_strategic_stop is non-null')

pit_rows['LapNumber_int'] = pd.to_numeric(pit_rows['LapNumber'], errors='coerce').round().astype('Int64')
pit_rows = pit_rows[pit_rows['LapNumber_int'].notna()].copy()

pit_true = pit_rows[pit_rows['is_strategic_stop'] == True]
pit_false = pit_rows[pit_rows['is_strategic_stop'] == False]

def lap_distribution_table(sub_df, label):
    counts = sub_df['LapNumber_int'].value_counts().sort_index().rename('count').to_frame()
    counts['pct'] = counts['count'] / counts['count'].sum() * 100.0
    counts.index.name = 'LapNumber'
    counts = counts.reset_index()
    counts['strategy_split'] = label
    return counts

lap_true_tbl = lap_distribution_table(pit_true, 'is_strategic_stop=True')
lap_false_tbl = lap_distribution_table(pit_false, 'is_strategic_stop=False')

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
for ax, sub_df, title, color in [
    (axes[0], pit_true, 'Pit stop lap distribution: is_strategic_stop=True', '#1f77b4'),
    (axes[1], pit_false, 'Pit stop lap distribution: is_strategic_stop=False', '#ff7f0e'),
]:
    ax.hist(sub_df['LapNumber_int'].astype(int), bins=40, color=color, alpha=0.85, edgecolor='black')
    ax.set_title(title)
    ax.set_ylabel('Count')
axes[1].set_xlabel('LapNumber')
plt.tight_layout()
pit_dist_fig = fig_dir / 'pit_stop_lap_distribution_by_strategy.png'
plt.savefig(pit_dist_fig, dpi=150)
plt.close()

print(f'Saved pit stop distribution plot: {pit_dist_fig}')

add_h2('2. Pit stop lap-number distribution split by is_strategic_stop')
add_p('Distribution table: is_strategic_stop=True')
add_table(lap_true_tbl)
add_p('Distribution table: is_strategic_stop=False')
add_table(lap_false_tbl)
add_p(f'Figure saved: {pit_dist_fig}')

# ---------------------------------------------------------------------
# 3) SC/VSC frequency by circuit (ranked table + bar chart)
# ---------------------------------------------------------------------
circuit_freq = (
    df.groupby('EventName', dropna=False)
      .agg(
          row_count=('EventName', 'size'),
          sc_frequency=('sc_active', 'mean'),
          vsc_frequency=('vsc_active', 'mean'),
          caution_frequency=('caution_active', 'mean')
      )
      .reset_index()
      .sort_values('caution_frequency', ascending=False)
)

fig, ax = plt.subplots(figsize=(14, 10))
plot_df = circuit_freq.sort_values('caution_frequency', ascending=True)
ax.barh(plot_df['EventName'], plot_df['caution_frequency'], color='#2ca02c')
ax.set_title('SC/VSC Caution Frequency by Circuit (2022-2024)')
ax.set_xlabel('Caution frequency (mean of caution_active)')
ax.set_ylabel('EventName')
plt.tight_layout()
sc_freq_fig = fig_dir / 'sc_vsc_frequency_by_circuit.png'
plt.savefig(sc_freq_fig, dpi=150)
plt.close()

print(f'Saved SC/VSC frequency chart: {sc_freq_fig}')

add_h2('3. SC/VSC frequency by circuit (2022-2024)')
add_table(circuit_freq)
add_p(f'Figure saved: {sc_freq_fig}')

# ---------------------------------------------------------------------
# 4) Archetype consistency check against assigned groups
# ---------------------------------------------------------------------
archetype_assignment = {
    'Street': ['Monaco Grand Prix', 'Azerbaijan Grand Prix', 'Singapore Grand Prix', 'Miami Grand Prix'],
    'Power': ['Italian Grand Prix', 'Belgian Grand Prix', 'British Grand Prix', 'Bahrain Grand Prix'],
    'Technical': ['Spanish Grand Prix', 'Hungarian Grand Prix', 'Abu Dhabi Grand Prix', 'Japanese Grand Prix'],
}

circuit_to_freq = dict(zip(circuit_freq['EventName'], circuit_freq['caution_frequency']))

assigned_rows = []
for archetype, circuits in archetype_assignment.items():
    vals = [circuit_to_freq.get(c, np.nan) for c in circuits]
    own_valid = np.array([v for v in vals if pd.notna(v)], dtype=float)
    own_median = float(np.median(own_valid)) if len(own_valid) else np.nan
    for c in circuits:
        v = circuit_to_freq.get(c, np.nan)
        assigned_rows.append({
            'assigned_archetype': archetype,
            'circuit': c,
            'empirical_caution_frequency': v,
            'own_archetype_median': own_median,
        })

archetype_check = pd.DataFrame(assigned_rows)

# Contradiction rule: circuit is closer to another archetype median than own median.
median_by_arch = {}
for archetype, circuits in archetype_assignment.items():
    vals = [circuit_to_freq.get(c, np.nan) for c in circuits]
    vals = [v for v in vals if pd.notna(v)]
    median_by_arch[archetype] = float(np.median(vals)) if vals else np.nan

def contradiction_flag(row):
    v = row['empirical_caution_frequency']
    own = row['assigned_archetype']
    if pd.isna(v) or pd.isna(median_by_arch.get(own, np.nan)):
        return 'MISSING_DATA'
    own_dist = abs(v - median_by_arch[own])
    other = {k: abs(v - m) for k, m in median_by_arch.items() if k != own and pd.notna(m)}
    if not other:
        return 'INSUFFICIENT_GROUP_DATA'
    nearest_other_arch = min(other, key=other.get)
    nearest_other_dist = other[nearest_other_arch]
    if nearest_other_dist + 1e-12 < own_dist:
        return f'CONTRADICTION -> closer to {nearest_other_arch}'
    return 'CONSISTENT'

archetype_check['consistency_flag'] = archetype_check.apply(contradiction_flag, axis=1)

add_h2('4. Archetype consistency check on empirical SC/VSC exposure')
add_p('Reference medians by assigned archetype (empirical caution_frequency):')
add_table(pd.Series(median_by_arch, name='median_caution_frequency'))
add_p('Circuit-level consistency results:')
add_table(archetype_check)

print('Archetype consistency check complete. Flagged contradictions:')
print(archetype_check[archetype_check['consistency_flag'].str.contains('CONTRADICTION', na=False)])

# ---------------------------------------------------------------------
# 5) Tyre degradation curves by compound, faceted by circuit
# ---------------------------------------------------------------------
curve_base = df[
    df['TyreCompound'].isin(['SOFT', 'MEDIUM', 'HARD'])
    & df['TyreLife'].notna()
    & df['fuel_corrected_laptime'].notna()
].copy()

# Require enough points for stable facet plots
min_points_per_facet = 120

for compound in ['SOFT', 'MEDIUM', 'HARD']:
    sub = curve_base[curve_base['TyreCompound'] == compound].copy()
    counts = sub['EventName'].value_counts()
    eligible_circuits = counts[counts >= min_points_per_facet].index.tolist()
    if not eligible_circuits:
        add_h3(f'5. {compound} degradation facets')
        add_p(f'No circuits met minimum sample size threshold ({min_points_per_facet}) for {compound}.')
        continue

    top_circuits = eligible_circuits[:9]
    fig, axes = plt.subplots(3, 3, figsize=(18, 14), sharex=False, sharey=False)
    axes = axes.flatten()

    for i, circuit in enumerate(top_circuits):
        ax = axes[i]
        csub = sub[sub['EventName'] == circuit].copy()
        x = pd.to_numeric(csub['TyreLife'], errors='coerce')
        y = pd.to_numeric(csub['fuel_corrected_laptime'], errors='coerce')
        mask = x.notna() & y.notna()
        x = x[mask]
        y = y[mask]

        ax.scatter(x, y, s=8, alpha=0.35)
        if len(x) >= 5:
            coeff = np.polyfit(x, y, 1)
            xx = np.linspace(x.min(), x.max(), 100)
            yy = coeff[0] * xx + coeff[1]
            ax.plot(xx, yy, color='red', linewidth=2)
            ax.text(0.02, 0.98, f'slope={coeff[0]:.4f}', transform=ax.transAxes, va='top')

        ax.set_title(f'{circuit}\n(n={len(x)})')
        ax.set_xlabel('TyreLife')
        ax.set_ylabel('fuel_corrected_laptime')

    for j in range(len(top_circuits), 9):
        axes[j].axis('off')

    fig.suptitle(f'{compound} degradation curves by circuit (top facets)', fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    fig_path = fig_dir / f'tyre_degradation_curves_{compound.lower()}.png'
    plt.savefig(fig_path, dpi=150)
    plt.close()

    add_h3(f'5. {compound} degradation facets')
    add_p(f'Figure saved: {fig_path}')

# ---------------------------------------------------------------------
# 6) Missing-data audit across all columns
# ---------------------------------------------------------------------
missing_audit = pd.DataFrame({
    'column': df.columns,
    'missing_count': [int(df[c].isna().sum()) for c in df.columns],
})
missing_audit['missing_pct'] = missing_audit['missing_count'] / len(df) * 100.0
missing_audit = missing_audit.sort_values('missing_pct', ascending=False).reset_index(drop=True)
missing_gt5 = missing_audit[missing_audit['missing_pct'] > 5.0].copy()

add_h2('6. Missing-data audit across all columns')
add_table(missing_audit)
add_p('Columns flagged above 5% missing:')
if missing_gt5.empty:
    add_p('None')
else:
    add_table(missing_gt5)

print('Missing-data audit complete. Columns >5% missing:')
print(missing_gt5)

# ---------------------------------------------------------------------
# 7) Outlier audit on six core feature groups (implemented per concrete column)
# ---------------------------------------------------------------------
outlier_cols = [
    'deg_rate_corrected',
    'pit_window_delta',
    'undercut_threat_index',
    'overcut_threat_index',
    'circuit_sc_rate',
    'circuit_vsc_rate',
    'pace_differential',
    'position_at_stake',
]

outlier_rows = []
for col in outlier_cols:
    s = pd.to_numeric(df[col], errors='coerce').dropna()
    n = len(s)
    if n == 0:
        outlier_rows.append({
            'feature': col,
            'non_null_n': 0,
            'outlier_3sd_count': np.nan,
            'outlier_3sd_pct': np.nan,
            'outlier_iqr_count': np.nan,
            'outlier_iqr_pct': np.nan,
        })
        continue

    mean = s.mean()
    std = s.std(ddof=0)
    if std == 0 or pd.isna(std):
        out_3sd = pd.Series(False, index=s.index)
    else:
        out_3sd = (s < mean - 3 * std) | (s > mean + 3 * std)

    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    if iqr == 0 or pd.isna(iqr):
        out_iqr = pd.Series(False, index=s.index)
    else:
        low = q1 - 1.5 * iqr
        high = q3 + 1.5 * iqr
        out_iqr = (s < low) | (s > high)

    outlier_rows.append({
        'feature': col,
        'non_null_n': int(n),
        'outlier_3sd_count': int(out_3sd.sum()),
        'outlier_3sd_pct': float(out_3sd.mean() * 100.0),
        'outlier_iqr_count': int(out_iqr.sum()),
        'outlier_iqr_pct': float(out_iqr.mean() * 100.0),
    })

outlier_df = pd.DataFrame(outlier_rows).sort_values('feature').reset_index(drop=True)

add_h2('7. Outlier audit on core engineered features')
add_p('Outliers are reported only (no dropping/transformation), using both ±3SD and IQR bounds.')
add_table(outlier_df)

print('Outlier audit complete:')
print(outlier_df)

# ---------------------------------------------------------------------
# Write plain-text markdown report
# ---------------------------------------------------------------------
report_path.write_text('\n'.join(report_lines), encoding='utf-8')
print(f'Wrote markdown report: {report_path}')

Descriptive statistics for engineered feature set:
                 feature       mean       std        min         max  \
0        circuit_sc_rate   0.114340  0.054988   0.025253    0.210526   
1       circuit_vsc_rate   0.025808  0.019931   0.000000    0.066976   
2     deg_rate_corrected   0.023348  0.053612  -1.157700    0.769737   
3   overcut_threat_index   6.220230  7.842950   0.000000   20.000000   
4      pace_differential   0.496219  4.272360 -59.557000  119.687000   
5       pit_window_delta  21.221150  3.098847  -4.116129   27.984132   
6      position_at_stake   0.498543  0.787575   0.000000    7.000000   
7  undercut_threat_index   7.486407  7.983825   0.000000   20.000000   

   missing_pct  
0     0.000000  
1     0.000000  
2    10.055544  
3     0.000000  
4     1.597886  
5    54.829881  
6    30.960878  
7     0.000000  
Saved pit stop distribution plot: /Users/demi/Downloads/Dissertation Code Project/Code Base/data/eda_outputs_phase1_base_df/figures/pit_stop_lap_di

In [4]:
# Optional: quick path check in notebook output
print('Report exists:', report_path.exists())
print('Figures folder exists:', fig_dir.exists())
print('Generated figure files:')
for p in sorted(fig_dir.glob('*.png')):
    print(' -', p.name)

Report exists: True
Figures folder exists: True
Generated figure files:
 - pit_stop_lap_distribution_by_strategy.png
 - sc_vsc_frequency_by_circuit.png
 - tyre_degradation_curves_hard.png
 - tyre_degradation_curves_hard_peryear.png
 - tyre_degradation_curves_medium.png
 - tyre_degradation_curves_medium_peryear.png
 - tyre_degradation_curves_soft.png
 - tyre_degradation_curves_soft_peryear.png
